# Model rollouts as training data

The consistency and enrichment results say two things: the constraints help, and asking for more reward does
enrich goal-directed moves -- by ~5 of a possible ~40 points at 10k steps. The ceiling on that is coverage:
of 66,609 far-start random walks in the data, **0** achieved their start's best outcome. Nothing can imitate
behaviour it has never seen.

So this manufactures it. For a training rollout, pick a switch time `tau`, keep the real prefix `h_tau`, and
let the model continue in R mode asking for **more** reward than the original got. The spliced trajectory
goes back into training -- teacher forcing, the value head, and the consistency term.

Three decisions, each of which the data's validity depends on:

- **Continuations step the real maze** (`maze.next_open`), not the model's dynamics head. Every spliced
  trajectory is a genuine trajectory of the environment; only the behaviour policy changes at `tau`.
- **R is relabelled to the bin the trajectory achieved**, never the one requested. Asking shapes which states
  get visited; the label follows what actually happened.
- **Every head trains on the mixture.** The interval identity is about one joint distribution. If only the R
  head saw rollouts while the NOR and value heads kept the random walk, the heads would model different
  joints and the consistency loss would fight the data. `train(mixer=...)` shuffles rollout rows into both
  halves of every batch and into the consistency batch.

A 2x2: `{mc, mc_all} x {data only, + rollouts}`, so the rollout effect and its interaction with consistency
both show up. The model stays at the size your 10k runs used, so `mc_all`'s lambda (calibrated there) still
holds and the numbers compare directly -- `D_MODEL` / `N_LAYERS` are one edit away if you want to scale it,
but re-check the consistency lambda if you do.

**Read section 3 first.** It is the direct test and needs no ground truth.

In [ ]:
REPO = "https://github.com/amdson/scrl.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
RUNS_DIR = "/content/drive/MyDrive/sillyrl/runs"  #@param {type:"string"}

In [ ]:
# Clone (or update) the repo and make sure the dependencies are importable.
import os, subprocess, sys

url = REPO
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret: public repo

if not os.path.exists("/content/sillyrl/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/sillyrl"], check=True)
else:
    subprocess.run(["git", "-C", "/content/sillyrl", "pull", "-q"], check=True)
os.chdir("/content/sillyrl")
sys.path.insert(0, "/content/sillyrl")

# Colab's preinstalled flax can lag its JAX (e.g. flax calling jax.core APIs that JAX 0.11 removed), so always
# upgrade flax and optax before importing them. pip keeps Colab's JAX if it already satisfies them; if it had
# to upgrade JAX, bring the GPU plugin (jax-cuda*) to the same version so the runtime doesn't fall back to CPU.
import importlib.metadata as md
jax_before = md.version("jax")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "flax", "optax"], check=True)
jax_after = md.version("jax")
if jax_after != jax_before:
    plugins = sorted({d.metadata["Name"] for d in md.distributions()
                      if (d.metadata["Name"] or "").lower().replace("_", "-").startswith("jax-cuda")})
    if plugins:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={jax_after}" for p in plugins]], check=True)
    print(f"jax {jax_before} -> {jax_after}; plugins updated: {plugins}")

import jax, flax, optax
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__, "|", jax.devices())

In [ ]:
# Where runs are saved. Must be set before importing maze_consistency.train.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RUNS_DIR"] = RUNS_DIR
else:
    os.environ["RUNS_DIR"] = "runs"
os.makedirs(os.environ["RUNS_DIR"], exist_ok=True)
print("runs ->", os.environ["RUNS_DIR"])

In [ ]:
# Rebuild the dataset (100k random walks) and the exact test set from data/canonical/maze.txt. Deterministic.
!python run.py dataset | tail -4
!python run.py testset | tail -1

## 1. Configuration

`SPECS` maps a run name to `(LossConfig, use rollouts?)`. Rollout settings:

- `MIX_FRAC` -- share of every main and consistency batch drawn from the rollout buffer
- `BUFFER_SIZE`, `REFRESH_EVERY` -- the buffer is regenerated from the current model this often
- `START_AFTER` -- no rollouts until the model has learned something worth rolling out
- `TAU_MAX` -- switch times are uniform on `0..TAU_MAX`, drawn independently of each trajectory's future
- `REQUEST` -- `"above"`: a bin uniformly above the original's, up to the best still achievable;
  `"best"`: always the best still achievable

In [ ]:
from dataclasses import replace
from maze_consistency.train import LossConfig
import maze_consistency.consistency as C

STEPS         = 20000    #@param {type:"integer"}
BATCH         = 32       #@param {type:"integer"}
D_MODEL       = 64       #@param {type:"integer"}
N_LAYERS      = 2        #@param {type:"integer"}
N_HEADS       = 4        #@param {type:"integer"}
MIX_FRAC      = 0.25     #@param {type:"number"}
BUFFER_SIZE   = 512      #@param {type:"integer"}
REFRESH_EVERY = 500      #@param {type:"integer"}
START_AFTER   = 2000     #@param {type:"integer"}
TAU_MAX       = 100      #@param {type:"integer"}
REQUEST       = "above"  #@param ["above", "best"]
PREFIX        = "rollouts"

cons = LossConfig(mc=True, cons=True, cons_loss="all", w_cons=0.011, cons_batch=16)
SPECS = {                                        # name: (LossConfig, use model rollouts?)
    "mc":          (LossConfig(mc=True), False),
    "mc_all":      (cons, False),
    "mc+roll":     (LossConfig(mc=True), True),
    "mc_all+roll": (cons, True),
}
LOSSES = {k: v[0] for k, v in SPECS.items()}     # the plotting helpers below key off LOSSES
for k, (lc, roll) in SPECS.items():
    print(f"  {k:<14} cons={lc.cons!s:<5} lambda={lc.w_cons if lc.cons else 0:<7g} rollouts={roll}")

## 2. The sweep

Same loop as the other notebooks, plus a `RolloutBuffer` for rollout runs. Each buffer's refresh history is
saved next to `history.json` as `buffer.json`. A rollout run prints one line per refresh:

    rollouts@2000: improved 0.31 (random walk 0.18)  got request 0.12 (rw 0.07)  reach 0.40 (orig 0.09) ...

Cost is not measured on a GPU yet: each refresh is ~200 forward passes over `BUFFER_SIZE` sequences on top of
training, so try `STEPS=3000` first to see the per-step time before committing to the full run.

In [ ]:
import os, json
import numpy as np
import jax.numpy as jnp
from maze_consistency.dataset import load as load_data
from maze_consistency.tokens import Tokenizer
from maze_consistency.model import ModelConfig, MazeTransformer, make_forward
from maze_consistency.testset import load_testset, stratified_rows, score
from maze_consistency.train import train, load_run, RUNS_DIR, N_HELDOUT
from maze_consistency.evaluate import make_enrichment_eval
from maze_consistency.augment import RolloutBuffer

MAZE, DATA = load_data()
TOK = Tokenizer(MAZE)
N_TRAIN = len(DATA["length"]) - N_HELDOUT
HELDOUT = np.random.default_rng(0).choice(np.arange(N_TRAIN, N_TRAIN + N_HELDOUT), 64, replace=False)


def run_sweep(specs, seeds=(0,), steps=STEPS, batch=BATCH, lr=1e-3, d_model=D_MODEL, n_layers=N_LAYERS,
              n_heads=N_HEADS, mix_frac=MIX_FRAC, buffer_size=BUFFER_SIZE, refresh_every=REFRESH_EVERY,
              start_after=START_AFTER, tau_max=TAU_MAX, request=REQUEST, eval_every=500,
              eval_per_setting=50, log_every=250, prefix=PREFIX, skip_existing=True, log=print):
    """specs: {run name: (LossConfig, use_rollouts)}. Rollout runs get their own RolloutBuffer."""
    ts = load_testset()
    rows = stratified_rows(ts, eval_per_setting, seed=0)
    model = MazeTransformer(ModelConfig.for_tokenizer(TOK, d_model=d_model, n_layers=n_layers, n_heads=n_heads))
    cons_eval = C.make_heldout_eval(model, TOK, MAZE, DATA, HELDOUT)
    enrich = make_enrichment_eval(TOK, MAZE, DATA, n=128)

    def eval_fn(params, fwd):
        m = score(params, fwd, TOK, ts, rows)
        m.pop("per_setting")
        m.update(cons_eval(params))
        m.update(enrich(params, fwd))            # enrich/* and goalward/*
        return m

    done = {}
    for name, (lc, use_rollouts) in specs.items():
        for seed in seeds:
            run = f"{prefix}/{name}_s{seed}"
            if skip_existing and os.path.exists(os.path.join(RUNS_DIR, run, "history.json")):
                log(f"[skip] {run} exists"); continue
            buf = (RolloutBuffer(model, TOK, MAZE, DATA, np.arange(N_TRAIN), size=buffer_size,
                                 refresh_every=refresh_every, start_after=start_after, tau_max=tau_max,
                                 request=request, seed=seed, log=log) if use_rollouts else None)
            done[run] = train(name=run, steps=steps, batch=batch, lr=lr, d_model=d_model, n_layers=n_layers,
                              n_heads=n_heads, seed=seed, loss=lc, eval_fn=eval_fn, eval_every=eval_every,
                              log_every=log_every, log=log, mixer=buf, mix_frac=mix_frac if buf else 0.0)
            if buf is not None:
                with open(os.path.join(RUNS_DIR, run, "buffer.json"), "w") as f:
                    json.dump(buf.history, f)
    return done


run_sweep(SPECS)

## 3. The direct test: asked for more, does the model get more?

Every refresh, from real prefixes, the model is asked for a better outcome than the data's own continuation
achieved. Each solid line is the model; each dotted line is the **exact** rate a uniform random walk would
manage from the same `(tau, s_tau)` (from the DP). That comparison is valid however training shifts the model,
so this is the panel to trust.

- **improved** -- beat the original continuation's bin
- **got request** -- reached at least the requested bin (the exact conditioned policy would score 100%)
- **reach** -- reached the goal at all, vs the original continuation
- **far own-best** -- far-start trajectories that hit their start's best bin. The data has **0** of these.
  Any nonzero count is behaviour the random-walk data never contained, now in the training set.

For `+roll` runs, the solid line pulling away from the dotted one over successive refreshes is the
self-improvement loop working.

In [ ]:
import matplotlib.pyplot as plt

def plot_buffer(prefix=PREFIX, seed=0, order=None):
    names = [n for n in (order or SPECS) if SPECS.get(n, (None, False))[1]]
    hist = {}
    for n in names:
        p = os.path.join(RUNS_DIR, prefix, f"{n}_s{seed}", "buffer.json")
        if os.path.exists(p):
            with open(p) as f:
                hist[n] = json.load(f)
    if not hist:
        print("no buffer.json yet -- rollout runs write one when they finish"); return hist
    panels = [("improved", "improved_rw", "beat the original continuation"),
              ("got_request", "got_request_rw", "reached at least the requested bin"),
              ("reach", "reach_orig", "reached the goal (dotted: original)"),
              ("far_own_best", None, "far starts hitting their own best bin (data: 0)")]
    fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
    for c, (n, h) in enumerate(hist.items()):
        steps = [r["step"] for r in h]
        for a, (key, ref, title) in zip(ax, panels):
            a.plot(steps, [r[key] for r in h], ".-", color=f"C{c}", label=n)
            if ref:
                a.plot(steps, [r[ref] for r in h], ":", color=f"C{c}", lw=1)
            a.set_title(title, fontsize=9); a.set_xlabel("step"); a.grid(alpha=.3)
    ax[0].legend(fontsize=7)
    fig.tight_layout()
    print(f"{'run':<14}{'improved':>10}{'(rw)':>7}{'got req':>9}{'(rw)':>7}{'reach':>8}{'(orig)':>8}{'far best':>10}")
    for n, h in hist.items():
        r = h[-1]
        print(f"{n:<14}{r['improved']:10.2f}{r['improved_rw']:7.2f}{r['got_request']:9.2f}"
              f"{r['got_request_rw']:7.2f}{r['reach']:8.2f}{r['reach_orig']:8.2f}{r['far_own_best']:10d}")
    return hist


_ = plot_buffer()
plt.show()

## 4. Exact-test metrics -- read with care for the rollout runs

`act_kl` and `value_kl` compare against the DP's answer **for the uniform random walk**. The `+roll` runs are
deliberately trained on a different distribution -- a mixture of the random walk and the model's own
goal-seeking continuations -- so their NOR and value heads are *supposed* to move away from the random-walk
truth. Worse numbers here for `+roll` are expected and are not a failure. Use them to compare the two
`data only` runs, and to check that nothing blows up.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt


def load_history(prefix, which="test"):
    """{run name: [history per seed]}. which="test" -> exact-test metrics at each checkpoint;
    which="train" -> logged loss parts, including cons / cond_gap / info_gain."""
    root = os.path.join(RUNS_DIR, prefix)
    out = {}
    for d in sorted(os.listdir(root)) if os.path.isdir(root) else []:
        p = os.path.join(root, d, "history.json")
        if os.path.exists(p):
            with open(p) as f:
                h = json.load(f)
            if not isinstance(h, dict) or which not in h:
                continue                      # pre-refactor runs stored a bare list; skip rather than crash
            out.setdefault(d.rsplit("_s", 1)[0], []).append(h[which])
    return out


def plot_curves(prefix, metrics=("act_kl", "value_kl"),
                settings=("NOR", "bin 10", "bin 11", "best far"), logy=True, order=None, figsize=(4.2, 3.4)):
    runs = load_history(prefix, "test")
    names = [n for n in (order or LOSSES) if n in runs] or list(runs)
    fig, ax = plt.subplots(len(metrics), len(settings),
                           figsize=(figsize[0] * len(settings), figsize[1] * len(metrics)), squeeze=False)
    for i, metric in enumerate(metrics):
        for j, setting in enumerate(settings):
            a, key = ax[i, j], f"{metric}/{setting}"
            for c, name in enumerate(names):
                hists = runs[name]
                steps = [m["step"] for m in hists[0]]
                ys = np.array([[m.get(key, np.nan) for m in h] for h in hists], dtype=float)
                a.plot(steps, np.nanmean(ys, 0), color=f"C{c}", label=name)
                if len(hists) > 1:
                    a.fill_between(steps, np.nanmin(ys, 0), np.nanmax(ys, 0), color=f"C{c}", alpha=.15)
            if logy:
                a.set_yscale("log")
            a.set_title(key, fontsize=9); a.set_xlabel("step"); a.grid(alpha=.3)
    ax[0, 0].legend(fontsize=8)
    fig.tight_layout()
    return fig


plot_curves(PREFIX)
plt.show()

## 5. Enrichment

Same paired probe as the other notebooks, on held-out random-walk prefixes: ask for bin 0 vs the best still
achievable from the identical prefix, and measure the gain in goalward mass. **Points** stay meaningful here.
The **% of exact** column divides by the random walk's own conditional -- a rollout-trained model learns a
mixture whose conditional can be sharper than that, so above 100% is possible and good.

In [ ]:
from maze_consistency.evaluate import make_enrichment_eval

def enrichment_table(prefix=PREFIX, names=None, seed=0, n=256):
    """Recompute enrichment from each run's saved params, so it works for older runs too."""
    probe = make_enrichment_eval(TOK, MAZE, DATA, n=n)
    names = [n_ for n_ in (names or list(LOSSES))]
    rows = {}
    for name in names:
        try:
            params, mcfg = load_run(f"{prefix}/{name}_s{seed}")
        except FileNotFoundError:
            continue
        rows[name] = probe(params, make_forward(MazeTransformer(mcfg), TOK))
    if not rows:
        print("no runs found for prefix", prefix); return rows
    w = max(len(k) for k in rows) + 2
    print(f"exact conditional gains {probe.ceiling:+.1f} points -- that is 100%")
    print(f"{'run':<{w}}{'ask: fail':>11}{'ask: best':>11}{'enrichment':>12}{'% of exact':>12}")
    for name, m in rows.items():
        print(f"{name:<{w}}{m['goalward/fail']:10.1f}%{m['goalward/best']:10.1f}%"
              f"{m['enrich/points']:+11.1f}{100*m['enrich/frac_of_exact']:11.0f}%")
    return rows


def enrichment_curves(prefix=PREFIX, order=None):
    """enrich/* over training, for runs that tracked it. Older runs simply do not appear."""
    runs = load_history(prefix, "test")
    names = [n for n in (order or LOSSES) if n in runs]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    drew = False
    for c, name in enumerate(names):
        hists = runs[name]
        if "enrich/points" not in hists[0][-1]:
            continue
        steps = [m["step"] for m in hists[0]]
        for j, key in enumerate(["enrich/points", "goalward/best"]):
            ys = np.array([[m.get(key, np.nan) for m in h] for h in hists], dtype=float)
            ax[j].plot(steps, np.nanmean(ys, 0), color=f"C{c}", label=name)
        ys = np.array([[m.get("goalward/fail", np.nan) for m in h] for h in hists], dtype=float)
        ax[1].plot(steps, np.nanmean(ys, 0), color=f"C{c}", ls=":", lw=1)
        drew = True
    if not drew:
        print("no run tracked enrich/* yet -- retrain, or use enrichment_table() on the checkpoints")
    ax[0].axhline(0, color="k", lw=.8, ls=":")
    ax[0].set_title("enrichment (points of goalward mass)", fontsize=9)
    ax[1].set_title("goalward mass: ask best (solid) vs ask fail (dotted)", fontsize=9)
    for a in ax:
        a.set_xlabel("step"); a.grid(alpha=.3); a.legend(fontsize=7)
    fig.tight_layout()
    return fig


_ = enrichment_table()
enrichment_curves()
plt.show()

## 6. Collapse diagnostics

`cond_gap` and `info_gain` for the consistency runs, as before. Training on the model's own outputs adds a
second way to collapse -- the model agreeing with itself -- so watch these more closely on `mc_all+roll`.

In [ ]:
def plot_diagnostics(prefix, keys=(("tf", "data: teacher-forced next-token loss", True),
                                   ("cons", "consistency objective", True),
                                   ("cond_gap", "cond_gap: mean |v_t - u_t|   (-> 0 = R ignored)", False),
                                   ("info_gain", "info_gain: log q_n(R) - log q_0(R)   (-> 0 = head flat)", False)),
                     order=None, figsize=(4.6, 3.8)):
    """Train-side view. keys is (history key, panel title, log y). Configs with no consistency term simply
    do not appear in the cons/cond_gap/info_gain panels."""
    runs = load_history(prefix, "train")
    names = [n for n in (order or LOSSES) if n in runs] or list(runs)
    fig, ax = plt.subplots(1, len(keys), figsize=(figsize[0] * len(keys), figsize[1]), squeeze=False)
    for j, (key, title, logy) in enumerate(keys):
        a = ax[0, j]
        for c, name in enumerate(names):
            hists = runs[name]
            steps = [m["step"] for m in hists[0]]
            ys = np.array([[m.get(key, np.nan) for m in h] for h in hists], dtype=float)
            if np.isnan(ys).all():
                continue
            a.plot(steps, np.nanmean(ys, 0), color=f"C{c}", label=name)
            if len(hists) > 1:
                a.fill_between(steps, np.nanmin(ys, 0), np.nanmax(ys, 0), color=f"C{c}", alpha=.15)
        a.set_yscale("log") if logy else a.axhline(0, color="k", lw=.8, ls=":")
        a.set_title(title, fontsize=9); a.set_xlabel("step"); a.grid(alpha=.3)
    ax[0, 0].legend(fontsize=7)
    fig.tight_layout()
    return fig


plot_diagnostics(PREFIX)
plt.show()

## Hacking this

- **Ask for the best, not just better**: `REQUEST = "best"`. More aggressive, fewer successes early.
- **More rollout data**: raise `MIX_FRAC` (0.5 is a reasonable upper test). Watch section 5 for the NOR head
  drifting and section 6 for collapse.
- **Rollouts only for consistency**: pass `mix_frac=0.0` and a `cons_sampler` that draws from the buffer.
  Cleaner for the heads -- the supervised targets stay the random walk -- but gives up the coverage that
  supervised rollout data provides, which is the point of this notebook.
- **Stay on-policy**: lower `REFRESH_EVERY`. Each refresh costs ~200 forward passes over `BUFFER_SIZE` rows.
- **No `td` with rollouts** -- its importance weight assumes the uniform behaviour policy, and `train`
  refuses the combination. `mc` and identity (A) are fine.